### Dataset Air BnB US dan Demografi US

In [1]:
import pandas as pd
import requests
import os

# Buat folder raw/ jika belum ada untuk penyimpanan log transaksi mentah
os.makedirs('raw', exist_ok=True)

def extract_etl_source1(file_path):
    print("--- Mengekstrak Sumber 1 (Airbnb) ---")
    if not os.path.exists(file_path):
        print(f"❌ ERROR: File '{file_path}' tidak ditemukan di Colab!")
        print("💡 Solusi: Mohon upload ulang file AB_US_2023.csv ke menu folder sebelah kiri.")
        return pd.DataFrame()

    try:
        df = pd.read_csv(file_path, low_memory=False)
        print(f"✅ Berhasil! Total data Airbnb mentah: {df.shape[0]} baris, {df.shape[1]} kolom")
        df.to_csv('raw/airbnb_raw_log.csv', index=False)
        return df
    except Exception as e:
        print("❌ Terjadi kesalahan saat membaca file CSV:", e)
        return pd.DataFrame()

def extract_etl_source2():
    print("\n--- Mengekstrak Sumber 2 (Demografi Kota via API) ---")
    # Menggunakan API Geonames Opendatasoft (Lebih stabil untuk tingkat Kota)
    api_url = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/geonames-all-cities-with-a-population-1000/records?where=cou_name_en%3D%22United%20States%22&limit=100"

    try:
        response = requests.get(api_url, timeout=10)
        if response.status_code == 200:
            data_json = response.json()
            if 'results' in data_json:
                df_demo = pd.DataFrame(data_json['results'])
                # Menyamakan nama kolom kota agar bisa di-join (Merge) dengan dataset Airbnb nanti
                if 'name' in df_demo.columns:
                    df_demo = df_demo.rename(columns={'name': 'city'})

                print(f"✅ Berhasil! Total data Demografi API: {df_demo.shape[0]} baris, {df_demo.shape[1]} kolom")
                df_demo.to_csv('raw/demografi_raw_log.csv', index=False)
                return df_demo
            else:
                print("⚠ Struktur JSON dari API berubah. Pindah ke mode Fallback Sintetis.")
        else:
            print(f"❌ API Gagal (Status {response.status_code}). Pindah ke mode Fallback Sintetis.")

    except Exception as e:
        print(f"❌ Koneksi API terputus ({e}). Pindah ke mode Fallback Sintetis.")

    # ==============================================================
    # FALLBACK SINTETIS (PENGAMAN): Dijamin tidak akan error lagi
    # Jika API gagal, program akan membuat data tiruan secara otomatis
    # ==============================================================
    print("🛡️ Mengaktifkan Data Demografi Sintetis untuk menyelamatkan Pipeline ETL...")
    dummy_data = pd.DataFrame({
        'city': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'San Diego', 'Dallas', 'Austin', 'Seattle', 'Denver', 'Boston', 'Miami'],
        'population': [8336817, 3971883, 2705994, 2320268, 1680992, 1423851, 1343573, 961855, 737015, 715522, 675647, 442241],
        'timezone': ['America/New_York', 'America/Los_Angeles', 'America/Chicago', 'America/Chicago', 'America/Phoenix', 'America/Los_Angeles', 'America/Chicago', 'America/Chicago', 'America/Los_Angeles', 'America/Denver', 'America/New_York', 'America/New_York']
    })
    dummy_data.to_csv('raw/demografi_raw_log.csv', index=False)
    return dummy_data

# Eksekusi fungsi ekstraksi data
df_airbnb_raw = extract_etl_source1('/content/AB_US_2023.csv')
df_demografi_raw = extract_etl_source2()

--- Mengekstrak Sumber 1 (Airbnb) ---
✅ Berhasil! Total data Airbnb mentah: 232147 baris, 18 kolom

--- Mengekstrak Sumber 2 (Demografi Kota via API) ---
✅ Berhasil! Total data Demografi API: 100 baris, 20 kolom


### Transform-Pembersihan Data

In [2]:
print("--- Proses Transform: Pembersihan & Standarisasi ---")

# 1. Duplikat data mentah agar data asli di Sel 1 tidak rusak
df_airbnb = df_airbnb_raw.copy()
df_demografi = df_demografi_raw.copy()

# 2. Mengubah nama kolom menjadi snake_case (huruf kecil semua, spasi diganti _)
df_airbnb.columns = [c.strip().lower().replace(' ', '_') for c in df_airbnb.columns]
df_demografi.columns = [c.strip().lower().replace(' ', '_') for c in df_demografi.columns]

# 3. PERBAIKAN MUTLAK: Menghapus baris duplikat spesifik berdasarkan Kolom ID Transaksi
df_airbnb = df_airbnb.drop_duplicates(subset=['id'])

# 4. Mengisi missing value pada kolom teks dan angka yang penting
df_airbnb['name'] = df_airbnb['name'].fillna('Unknown')
df_airbnb['host_name'] = df_airbnb['host_name'].fillna('Unknown')
df_airbnb['reviews_per_month'] = df_airbnb['reviews_per_month'].fillna(0)

# 5. Mengubah format tanggal last_review menjadi Datetime asli pandas
df_airbnb['last_review'] = pd.to_datetime(df_airbnb['last_review'], errors='coerce')

# 6. Menangani outlier harga dengan metode IQR (Batas Atas & Bawah)
Q1 = df_airbnb['price'].quantile(0.25)
Q3 = df_airbnb['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Memfilter data di dalam rentang IQR yang logis
df_airbnb_clean = df_airbnb[(df_airbnb['price'] >= lower_bound) & (df_airbnb['price'] <= upper_bound)].copy()
print(f"Data Airbnb berhasil dibersihkan.")
print(f"Sisa data setelah pembuangan outlier & ID duplikat: {df_airbnb_clean.shape[0]} baris")

--- Proses Transform: Pembersihan & Standarisasi ---
Data Airbnb berhasil dibersihkan.
Sisa data setelah pembuangan outlier & ID duplikat: 212637 baris


### Transform-Penggabungan & Rekayasa

In [3]:
print("--- Proses Transform: Enrichment, Feature Engineering, Normalisasi & Encoding ---")

# Standardisasi penulisan nama kota (Capitalize) agar proses Merge sukses 100%
df_airbnb_clean['city'] = df_airbnb_clean['city'].astype(str).str.title()

if 'city' in df_demografi.columns:
    df_demografi['city'] = df_demografi['city'].astype(str).str.title()
    # Hapus duplikat kota pada data demografi agar tidak menyebabkan perluasan baris (Explosion)
    df_demografi_unique = df_demografi.drop_duplicates(subset=['city'])
else:
    # Jika API bermasalah, buat dataframe kosong dengan kolom kota sebagai penjamin flow kode
    df_demografi_unique = pd.DataFrame(columns=['city'])

# Lakukan Left Join data Airbnb dengan Demografi US berdasarkan Kota
df_enriched = pd.merge(df_airbnb_clean, df_demografi_unique, on='city', how='left')

# ================= 1. MEMBUAT 5 FITUR BARU =================

# Fitur 1: price_category (Membagi kelompok harga menjadi Murah, Sedang, Mahal)
df_enriched['price_category'] = pd.qcut(df_enriched['price'], q=3, labels=['Murah', 'Sedang', 'Mahal'])

# Fitur 2: review_engagement (Mengukur keaktifan ulasan konsumen)
df_enriched['review_engagement'] = df_enriched['number_of_reviews'].apply(lambda x: 'Tinggi' if x > 50 else 'Rendah')

# Fitur 3: is_highly_available (Penanda ketersediaan jangka panjang jika ketersediaan >= 180 hari)
df_enriched['is_highly_available'] = df_enriched['availability_365'].apply(lambda x: 1 if x >= 180 else 0)

# Fitur 4: total_estimated_revenue (Estimasi pendapatan minimum: harga per malam * syarat minimum menginap)
df_enriched['total_estimated_revenue'] = df_enriched['price'] * df_enriched['minimum_nights']

# Fitur 5: host_dominance (Mendeteksi apakah host termasuk skala bisnis besar/Dominan atau Normal)
df_enriched['host_dominance'] = df_enriched['calculated_host_listings_count'].apply(lambda x: 'Dominant' if x >= 5 else 'Normal')


# ================= 2. TAMBAHAN UPGRADE: NORMALISASI (Min-Max Scaling) =================
# Sesuai syarat Bab 6.2.b: Melakukan normalisasi pada minimal dua kolom numerik (price & minimum_nights)
# Mengubah rentang nilai angka menjadi skala 0 sampai 1 agar adil saat dianalisis
df_enriched['normalized_price'] = (df_enriched['price'] - df_enriched['price'].min()) / (df_enriched['price'].max() - df_enriched['price'].min())
df_enriched['normalized_minimum_nights'] = (df_enriched['minimum_nights'] - df_enriched['minimum_nights'].min()) / (df_enriched['minimum_nights'].max() - df_enriched['minimum_nights'].min())


# ================= 3. TAMBAHAN UPGRADE: ENCODING KATEGORIKAL =================
# Sesuai syarat Bab 6.2.b: Melakukan encoding pada kolom kategorikal (room_type & price_category)
# Mengubah teks kategori menjadi kode angka agar bisa dibaca optimal oleh sistem warehouse
df_enriched['room_type_encoded'] = df_enriched['room_type'].astype('category').cat.codes
df_enriched['price_category_encoded'] = df_enriched['price_category'].astype('category').cat.codes

print("✅ Sukses! 5 Fitur baru dibuat, 2 Kolom dinormalisasi, & Kolom kategori berhasil di-encode.")
# Menampilkan cuplikan kolom baru hasil normalisasi dan encoding
display(df_enriched[['price', 'normalized_price', 'minimum_nights', 'normalized_minimum_nights', 'room_type', 'room_type_encoded']].head())

--- Proses Transform: Enrichment, Feature Engineering, Normalisasi & Encoding ---
✅ Sukses! 5 Fitur baru dibuat, 2 Kolom dinormalisasi, & Kolom kategori berhasil di-encode.


,price,normalized_price,minimum_nights,normalized_minimum_nights,room_type,room_type_encoded
0,202,0.413934,2,0.000801,Entire home/apt,0
1,235,0.481557,30,0.023219,Entire home/apt,0
2,56,0.114754,32,0.024820,Private room,2
3,110,0.225410,1,0.000000,Private room,2
4,95,0.194672,1,0.000000,Private room,2


Transform-Validasi Kualitas Data

In [4]:
print("--- Proses Transform: Validasi Kualitas Data (6 Aturan) ---")
failed_rules = 0

# Aturan 1: Memastikan ID transaksi unik
if df_enriched['id'].duplicated().any():
    print("❌ Aturan 1 Gagal: Ditemukan duplikasi pada Kolom ID.")
    failed_rules += 1
else:
    print("✅ Aturan 1 Lulus: Seluruh ID unik.")

# Aturan 2: Validasi Harga Properti
if (df_enriched['price'] < 0).any():
    print("❌ Aturan 2 Gagal: Terdapat harga bernilai minus (negatif).")
    failed_rules += 1
else:
    print("✅ Aturan 2 Lulus: Nilai harga valid (tidak ada minus).")

# Aturan 3: Tipe Data Batas Menginap
if not pd.api.types.is_numeric_dtype(df_enriched['minimum_nights']):
    print("❌ Aturan 3 Gagal: Tipe data minimum_nights tidak seragam numerik.")
    failed_rules += 1
else:
    print("✅ Aturan 3 Lulus: Tipe data minimum_nights konsisten berupa angka.")

# Aturan 4: Batas Logika Ketersediaan Tahunan
if (df_enriched['availability_365'] > 365).any():
    print("❌ Aturan 4 Gagal: Nilai availability melanggar batas logika kalender (> 365 hari).")
    failed_rules += 1
else:
    print("✅ Aturan 4 Lulus: Rentang hari ketersediaan masuk akal.")

# Aturan 5: Kelengkapan Data Wilayah
if df_enriched['city'].isnull().any():
    print("❌ Aturan 5 Gagal: Terdapat data transaksi yang kehilangan informasi nama kota (Null).")
    failed_rules += 1
else:
    print("✅ Aturan 5 Lulus: Data spasial kota lengkap.")

# Aturan 6: Logika Listing Kepemilikan Host
if (df_enriched['calculated_host_listings_count'] < 1).any():
    print("❌ Aturan 6 Gagal: Jumlah listing yang dimiliki host tidak valid (kurang dari 1).")
    failed_rules += 1
else:
    print("✅ Aturan 6 Lulus: Jumlah kepemilikan aset host valid.")

print("-" * 50)
if failed_rules == 0:
    print("🎉 DATA 100% VALID DAN BERSIH! Siap di-load ke Data Warehouse.")
else:
    print(f"⚠ Perhatian: Terdapat {failed_rules} aturan yang gagal dipenuhi. Periksa kembali logika pembersihan.")

--- Proses Transform: Validasi Kualitas Data (6 Aturan) ---
✅ Aturan 1 Lulus: Seluruh ID unik.
✅ Aturan 2 Lulus: Nilai harga valid (tidak ada minus).
✅ Aturan 3 Lulus: Tipe data minimum_nights konsisten berupa angka.
✅ Aturan 4 Lulus: Rentang hari ketersediaan masuk akal.
✅ Aturan 5 Lulus: Data spasial kota lengkap.
✅ Aturan 6 Lulus: Jumlah kepemilikan aset host valid.
--------------------------------------------------
🎉 DATA 100% VALID DAN BERSIH! Siap di-load ke Data Warehouse.


Load-Pembentukan Arsitektur Star Schema

In [5]:
import os
import sqlite3
print("--- Proses Load: Menghasilkan Arsitektur Star Schema & Database SQL ---")

# 1. Buat direktori lokal penyimpanan Data Warehouse berkas
os.makedirs('data_warehouse', exist_ok=True)

# 2. Inisialisasi koneksi ke Database SQLite (Gudang Data Relasional)
db_path = 'data_warehouse/airbnb_warehouse.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# -------------------------------------------------------------
# A. PEMBENTUKAN TABEL DIMENSI & FAKTA (FORMAT CSV)
# -------------------------------------------------------------

# Dimensi 1: dim_host (Profil Unik Host)
dim_host = df_enriched[['host_id', 'host_name', 'calculated_host_listings_count', 'host_dominance']].drop_duplicates(subset=['host_id'])
dim_host.to_csv('data_warehouse/dim_host.csv', index=False)

# Dimensi 2: dim_city (Kombinasi Geografis & Demografi Kota hasil API)
demo_cols = [col for col in df_demografi_unique.columns if col != 'city' and col in df_enriched.columns]
dim_city = df_enriched[['city'] + demo_cols].drop_duplicates(subset=['city'])
dim_city.to_csv('data_warehouse/dim_city.csv', index=False)

# Dimensi 3: dim_time (Aspek Waktu Terperinci dari Tanggal Review)
dim_time = df_enriched[['last_review']].drop_duplicates().dropna().copy()
dim_time['year'] = dim_time['last_review'].dt.year
dim_time['month'] = dim_time['last_review'].dt.month
dim_time['day'] = dim_time['last_review'].dt.day
dim_time.to_csv('data_warehouse/dim_time.csv', index=False)

# Tabel Fakta Utama: fact_airbnb (Pusat Transaksi Bisnis)
fact_airbnb = df_enriched[['id', 'host_id', 'city', 'last_review', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'price_category', 'total_estimated_revenue', 'is_highly_available']]
fact_airbnb.to_csv('data_warehouse/fact_airbnb.csv', index=False)


# -------------------------------------------------------------
# B. PROSES LOAD KE DATABASE RELASIONAL SQLITE (Syarat Bab 6.3)
# -------------------------------------------------------------
print("\n🔄 Memulai proses loading data ke dalam Database Relasional...")

# Memasukkan DataFrame langsung menjadi tabel SQL di dalam Database
dim_host.to_sql('dim_host', conn, if_exists='replace', index=False)
dim_city.to_sql('dim_city', conn, if_exists='replace', index=False)

# Mengonversi kolom tanggal di DataFrame waktu menjadi string agar aman di SQLite
dim_time_sql = dim_time.copy()
dim_time_sql['last_review'] = dim_time_sql['last_review'].dt.strftime('%Y-%m-%d')
dim_time_sql.to_sql('dim_time', conn, if_exists='replace', index=False)

fact_airbnb_sql = fact_airbnb.copy()
fact_airbnb_sql['last_review'] = fact_airbnb_sql['last_review'].dt.strftime('%Y-%m-%d')
fact_airbnb_sql.to_sql('fact_airbnb', conn, if_exists='replace', index=False)

# Commit perubahan dan tutup koneksi database sementara
conn.commit()
conn.close()

print("✅ PROSES LOAD SELESAI SEMPURNA!")
print(f"Gudang data SQL berhasil diciptakan di lokasi: '{db_path}'")
print("Struktur Relasional Star Schema Anda di DB mencakup:")
print("  1. Table 'fact_airbnb' (Tabel Fakta Utama)")
print("  2. Table 'dim_host' (Dimensi Kepemilikan Host)")
print("  3. Table 'dim_city' (Dimensi Geografis Kota)")
print("  4. Table 'dim_time' (Dimensi Kalender Tren)")

--- Proses Load: Menghasilkan Arsitektur Star Schema & Database SQL ---

🔄 Memulai proses loading data ke dalam Database Relasional...
✅ PROSES LOAD SELESAI SEMPURNA!
Gudang data SQL berhasil diciptakan di lokasi: 'data_warehouse/airbnb_warehouse.db'
Struktur Relasional Star Schema Anda di DB mencakup:
  1. Table 'fact_airbnb' (Tabel Fakta Utama)
  2. Table 'dim_host' (Dimensi Kepemilikan Host)
  3. Table 'dim_city' (Dimensi Geografis Kota)
  4. Table 'dim_time' (Dimensi Kalender Tren)


### Verifikasi Data Warehouse

In [6]:
import sqlite3
import pandas as pd

print("--- Menjalankan 8 Query SQL Analitik pada Data Warehouse ---")

# 1. Hubungkan ke database gudang data yang sudah dibuat di Sel 5
db_path = 'data_warehouse/airbnb_warehouse.db'
conn = sqlite3.connect(db_path)

# Fungsi pembantu untuk menjalankan query dan langsung menampilkannya dengan rapi
def jalankan_analisis_sql(nomor_query, deskripsi, query_string):
    print(f"\n=======================================================")
    print(f"📊 QUERY {nomor_query}: {deskripsi}")
    print(f"=======================================================")
    df_hasil = pd.read_sql_query(query_string, conn)
    display(df_hasil)

# --- 8 QUERY ANALITIK YANG DIWAJIBKAN DOSEN ---

# Query 1: Menghitung total transaksi (baris) untuk memastikan kelengkapan Load data
jalankan_analisis_sql(
    1, "Total Baris Data pada Tabel Fakta (fact_airbnb)",
    "SELECT COUNT(*) AS total_records_loaded FROM fact_airbnb;"
)

# Query 2: Rata-rata harga sewa berdasarkan Kategori Harga (Fitur Turunan 1)
jalankan_analisis_sql(
    2, "Rata-rata Harga Asli Berdasarkan Kelompok Kategori Harga",
    """SELECT price_category, COUNT(*) as jumlah_properti, ROUND(AVG(price), 2) as rata_rata_harga
       FROM fact_airbnb
       GROUP BY price_category;"""
)

# Query 3: Top 5 Kota dengan akumulasi estimasi pendapatan kotor tertinggi (Fitur Turunan 4)
jalankan_analisis_sql(
    3, "Top 5 Kota dengan Estimasi Pendapatan Terbesar (Price * Min Nights)",
    """SELECT city, SUM(total_estimated_revenue) as total_pendapatan_kota
       FROM fact_airbnb
       GROUP BY city
       ORDER BY total_pendapatan_kota DESC
       LIMIT 5;"""
)

# Query 4: Analisis dominansi kepemilikan host terhadap keaktifan ulasan konsumen (Fitur Turunan 5)
jalankan_analisis_sql(
    4, "Hubungan Skala Bisnis Host (Dominance) dengan Total Ulasan Masuk",
    """SELECT h.host_dominance, COUNT(f.id) as total_properti, SUM(f.number_of_reviews) as total_ulasan_diperoleh
       FROM fact_airbnb f
       JOIN dim_host h ON f.host_id = h.host_id
       GROUP BY h.host_dominance;"""
)

# Query 5: Distribusi tipe kamar beserta rata-rata harganya dan tingkat ketersediaan jangka panjang (Fitur Turunan 3)
jalankan_analisis_sql(
    5, "Analisis Karakteristik Tipe Kamar Airbnb",
    """SELECT room_type, COUNT(*) as jumlah_listing, ROUND(AVG(price), 2) as rata_rata_harga, SUM(is_highly_available) as siap_sewa_tahunan
       FROM fact_airbnb
       GROUP BY room_type;"""
)

# Query 6: Tren rata-rata harga transaksi akomodasi berdasarkan tahun review (Menggunakan Dimensi Waktu)
jalankan_analisis_sql(
    6, "Tren Fluktuasi Rata-rata Harga Sewa Berdasarkan Tahun Catatan Ulasan",
    """SELECT t.year, COUNT(f.id) as volume_transaksi, ROUND(AVG(f.price), 2) as rata_rata_harga_tahunan
       FROM fact_airbnb f
       JOIN dim_time t ON f.last_review = t.last_review
       GROUP BY t.year
       ORDER BY t.year ASC;"""
)

# Query 7: Penggabungan Spasial & Demografi hasil penarikan API (Enrichment Data Sekunder)
jalankan_analisis_sql(
    7, "Analisis Korelasi Populasi Kota (Data API) dengan Tingkat Kerapatan Properti Airbnb",
    """SELECT f.city, c.population as populasi_kota_api, COUNT(f.id) as jumlah_airbnb_terdaftar
       FROM fact_airbnb f
       JOIN dim_city c ON f.city = c.city
       WHERE c.population IS NOT NULL
       GROUP BY f.city, c.population
       ORDER BY jumlah_airbnb_terdaftar DESC
       LIMIT 5;"""
)

# Query 8: Rasio properti berketersediaan tinggi di tiap segmen kelas ekonomi
jalankan_analisis_sql(
    8, "Persentase Properti Siap Sewa Jangka Panjang (>180 Hari) per Kategori Harga",
    """SELECT price_category, COUNT(*) as total_properti, SUM(is_highly_available) as jumlah_high_available,
       ROUND((CAST(SUM(is_highly_available) AS FLOAT) / COUNT(*)) * 100, 2) || '%' as persentase_rasio
       FROM fact_airbnb
       GROUP BY price_category;"""
)

# Tutup koneksi secara aman setelah seluruh verifikasi selesai
conn.close()
print("\n🎉 SELURUH 8 QUERY BERHASIL DIEKSEKUSI DAN VALIDASI WAREHOUSE SELESAI!")

--- Menjalankan 8 Query SQL Analitik pada Data Warehouse ---

📊 QUERY 1: Total Baris Data pada Tabel Fakta (fact_airbnb)


,total_records_loaded
0,212637



📊 QUERY 2: Rata-rata Harga Asli Berdasarkan Kelompok Kategori Harga


,price_category,jumlah_properti,rata_rata_harga
0,Mahal,70502,279.38
1,Murah,72108,70.05
2,Sedang,70027,138.94



📊 QUERY 3: Top 5 Kota dengan Estimasi Pendapatan Terbesar (Price * Min Nights)


...


📊 QUERY 7: Analisis Korelasi Populasi Kota (Data API) dengan Tingkat Kerapatan Properti Airbnb


,city,populasi_kota_api,jumlah_airbnb_terdaftar



📊 QUERY 8: Persentase Properti Siap Sewa Jangka Panjang (>180 Hari) per Kategori Harga


,price_category,total_properti,jumlah_high_available,persentase_rasio
0,Mahal,70502,36570,51.87%
1,Murah,72108,30593,42.43%
2,Sedang,70027,33846,48.33%



🎉 SELURUH 8 QUERY BERHASIL DIEKSEKUSI DAN VALIDASI WAREHOUSE SELESAI!
